In [ ]:
# --- Setup: preload DAM, load KB ---
import sys, time
from pathlib import Path
sys.path.insert(0, "..")

from hanoi_caption.dam_model import MODEL_NAME as DAM_NAME
from hanoi_caption.kb_loader import load_kb
from hanoi_caption.model_registry import registry

nodes = load_kb("../data/kb.json")
print(f"KB ready: {len(nodes)} nodes")

t0 = time.perf_counter()
registry.get(DAM_NAME)
print(f"DAM loaded in {time.perf_counter()-t0:.1f}s")


In [ ]:
# --- Video pipeline demo: run caption_video on a real clip ---
import time
from pathlib import Path

from hanoi_caption.video_pipeline import caption_video

VIDEO_PATH = Path("../tests/fixtures/video/A39_203_HTTLCongDoanMonTrai_M_T02.MOV")

if not VIDEO_PATH.exists():
    print(f"Drop a video at: {VIDEO_PATH.resolve()}")
    print("(A 10-60s walking-tour clip showing 1-3 Hanoi landmarks works well.)")
    segments = []
else:
    print(f"Processing {VIDEO_PATH.name} ...")
    t0 = time.perf_counter()
    segments = caption_video(
        video_path=VIDEO_PATH,
        kb_nodes=nodes,
        dino_index_path="../data/cache/dino_faiss.index",
        id_map_path="../data/cache/id_map.json",
        sample_fps=1.0,
        smooth_window=3,
        min_segment_seconds=2.0,
        dam_frame_budget=(4, 8),
    )
    elapsed = time.perf_counter() - t0
    print(f"Done in {elapsed:.1f}s -> {len(segments)} segment(s)")


In [ ]:
# --- Render segments: timeline plot + caption list ---
import matplotlib.pyplot as plt

if not segments:
    print("No segments to render. Run the previous cell first.")
else:
    # Unique landmarks in order of appearance.
    seen = {}
    for s in segments:
        if s.kb_id not in seen:
            seen[s.kb_id] = s.name_en
    unique_kb_ids = list(seen.keys())
    cmap = plt.get_cmap("tab10")
    color_by_kb_id = {kb_id: cmap(i % 10) for i, kb_id in enumerate(unique_kb_ids)}

    # Timeline: one row per landmark, broken_barh shows segment spans.
    fig, ax = plt.subplots(figsize=(12, 1.0 + 0.55 * len(unique_kb_ids)))
    for i, kb_id in enumerate(unique_kb_ids):
        spans = [(s.start_s, s.end_s - s.start_s) for s in segments if s.kb_id == kb_id]
        ax.broken_barh(spans, (i - 0.4, 0.8),
                       facecolors=color_by_kb_id[kb_id], edgecolor="black", linewidth=0.5)

    ax.set_yticks(range(len(unique_kb_ids)))
    ax.set_yticklabels([seen[k] for k in unique_kb_ids])
    ax.invert_yaxis()  # first landmark on top
    ax.set_xlabel("Time (s)")
    ax.set_xlim(0, max(s.end_s for s in segments) * 1.02)
    ax.set_title(f"Landmark segments in {VIDEO_PATH.name}", fontweight="bold")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Captions list.
    for i, s in enumerate(segments, 1):
        print("=" * 80)
        print(f"Segment {i}: {s.start_s:.1f}s - {s.end_s:.1f}s  ({s.end_s - s.start_s:.1f}s)")
        print(f"Landmark : {s.name_en}  (confidence {s.confidence:.3f})")
        print(f"Frames   : {s.debug.get('frames_sampled', '?')}/{s.debug.get('frames_total', '?')}  "
              f"DAM: {s.debug.get('timings', {}).get('dam_caption', 0):.1f}s")
        print("-" * 80)
        print(s.caption)
    print("=" * 80)
